#BRONZE INGESTION — E-Commerce Lakehouse

In [0]:
from pyspark.sql.functions import *

In [ ]:
df = spark.read.format("csv")\
        .option("header", True)\
            .option("inferSchema", True)\
                .load("/Volumes/ecommerce_lakehouse/default/source_data_csv/olist_orders_dataset.csv")

In [0]:
df.limit(5).display()

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("_load_timestamp", current_timestamp())\
          .withColumn("_source_file", col("_metadata.file_path"))

df.limit(5).display()
    

## Configs

In [ ]:
base_path = "/Volumes/ecommerce_lakehouse/default/source_data_csv"

file_to_table = {
    "olist_orders_dataset.csv":        "orders",
    "olist_order_items_dataset.csv":   "order_items",
    "olist_customers_dataset.csv":     "customers",
    "olist_products_dataset.csv":      "products",
    "olist_order_payments_dataset.csv":"payments",
    # "olist_order_reviews_dataset.csv": "reviews",
}

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.bronze")

for file_name, table_name in file_to_table.items():
    source = f"{base_path}/{file_name}"

    # 1. Read raw — schema-on-read, capture as-is
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(source)
    )

    # 2. Add lineage IMMEDIATELY, while file context is still available.
    #    _metadata.file_path works on Unity Catalog; input_file_name() is
    #    use the built-in _metadata column.
    df = (
        df.withColumn("_load_timestamp", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
    )

    # 3. Write to Delta. overwrite => deterministic re-runs on a static dataset.
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .load(source)
    )

    print(f"ecommerce_lakehouse.bronze.{table_name}  ←  {file_name}  ({df.count():,} rows)")

print("\nBronze ingestion complete.")

In [ ]:
display(spark.table("ecommerce_lakehouse.bronze.orders").select("order_id", "_load_timestamp", "_source_file").limit(5))


## Bronze ingestion of Review as because Reviews CSV have some garbage data

In [0]:
from pyspark.sql.types import *

In [0]:
reviews_schema = StructType([
    StructField("review_id", StringType()),
    StructField("order_id", StringType()),
    StructField("review_score", IntegerType()),
    StructField("review_comment_title", StringType()),
    StructField("review_comment_message", StringType()),
    StructField("review_creation_date", TimestampType()),
    StructField("review_answer_timestamp", TimestampType()),
    StructField("_corrupt_record", StringType()), 
])

In [ ]:
base_path = "/Volumes/ecommerce_lakehouse/default/source_data_csv"


df_reviews = (
    spark.read.format("csv")
    .option("header", True)
    .option("multiLine", True)              # records can span physical lines
    .option("quote", '"')                   # " wraps a field
    .option("escape", '"')                  # "" inside a field = one literal "  ← the key fix
    .option("mode", "PERMISSIVE")           # don't crash on bad rows
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(reviews_schema)                 # see below — explicit schema matters here
    .load(f"{base_path}/olist_order_reviews_dataset.csv")
)
df_reviews.count()

In [0]:
df_reviews.display()

In [0]:
clean = df_reviews.filter(col("_corrupt_record").isNull())    
clean.count()

In [0]:
clean = clean.filter(length(col("review_id")) == 32)              # valid id shape
clean.count()

In [0]:
clean = clean.drop("_corrupt_record")

In [0]:
clean.limit(5).display()

### Write to Bronze Table

In [ ]:
df_reviews.write.format("delta")\
    .mode("overwrite")\
        .option("overwriteSchema", "true")\
            .saveAsTable("ecommerce_lakehouse.bronze.reviews")

In [ ]:
spark.table("ecommerce_lakehouse.bronze.reviews").limit(5).display()